# Transformação Silver - Pesquisa de Satisfação

## Descrição
Este notebook realiza a transformação de dados da camada **Bronze** para **Silver** da tabela `pesquisa_satisfacao`, incluindo limpeza, engenharia de features e garantia de qualidade.

## Objetivos
- Renomear colunas para nomes descritivos
- Converter tipo de dados da coluna `nota_atendimento`
- Tratar valores nulos (substituindo pela mediana)
- Criar categoria de nota (Insatisfeito, Neutro, Satisfeito)
- Salvar na camada Silver em formato Delta Lake


### Importando bibliotecas e carregando tabela

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

catalogo = "medalhao_credit"
silver_db_name = "silver_credit"
path_pesquisa = "medalhao_credit.bronze_credit.pesquisa_satisfacao"

df_bronze = spark.read.table(path_pesquisa)
display(df_bronze.limit(5))

_c0,_c1,_c2,data_ingestao
1,1,3,2025-11-21T14:38:41.734Z
2,2,2,2025-11-21T14:38:41.734Z
3,3,3,2025-11-21T14:38:41.734Z
4,4,3,2025-11-21T14:38:41.734Z
6,5,2,2025-11-21T14:38:41.734Z


### Análise Exploratória
Neste trecho iremos analisar o schema, uma amostra da tabela bem como suas estatísticas com a função describe, por fim, iremos ver a distribuição das notas.


In [0]:
print(f"Total de registros: {df_bronze.count()}")

# Schema e amostra
df_bronze.printSchema()
display(df_bronze.limit(10))

# Estatísticas das notas
display(df_bronze.describe())

# Distribuição das notas
display(df_bronze.groupBy("_c2").count().orderBy("_c2"))

Total de registros: 30000
root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)



_c0,_c1,_c2,data_ingestao
1,1,3,2025-11-21T14:38:41.734Z
2,2,2,2025-11-21T14:38:41.734Z
3,3,3,2025-11-21T14:38:41.734Z
4,4,3,2025-11-21T14:38:41.734Z
6,5,2,2025-11-21T14:38:41.734Z
7,6,3,2025-11-21T14:38:41.734Z
8,7,2,2025-11-21T14:38:41.734Z
9,8,2,2025-11-21T14:38:41.734Z
10,9,3,2025-11-21T14:38:41.734Z
158,148,2,2025-11-21T14:38:41.734Z


summary,_c0,_c1,_c2
count,30000,30000,30000
mean,15000.5,15000.5,2.97985347985348
stddev,8660.398374208868,8660.39837420882,0.8392745820170685
min,1,1,2
max,30000,30000,NULL


_c2,count
2,8530
3,12155
4,5250
5,1365
NULL,2700


#### Problemas Identificados
1. **Nomes de colunas não descritivos**
2. **Tipo incorreto**: `nota_atendimento` como string
3. **Valores nulos**: 2.700 registros

### Transformações Aplicadas

#### 1. Renomeação de Colunas

In [0]:
df_bronze = (df_bronze
    .withColumnRenamed("_c0", "id_pesquisa")
    .withColumnRenamed("_c1", "id_chamado") 
    .withColumnRenamed("_c2", "nota_atendimento")
)

display(df_bronze.limit(10))

id_pesquisa,id_chamado,nota_atendimento,data_ingestao
1,1,3,2025-11-21T14:38:41.734Z
2,2,2,2025-11-21T14:38:41.734Z
3,3,3,2025-11-21T14:38:41.734Z
4,4,3,2025-11-21T14:38:41.734Z
6,5,2,2025-11-21T14:38:41.734Z
7,6,3,2025-11-21T14:38:41.734Z
8,7,2,2025-11-21T14:38:41.734Z
9,8,2,2025-11-21T14:38:41.734Z
10,9,3,2025-11-21T14:38:41.734Z
158,148,2,2025-11-21T14:38:41.734Z


#### 2. Conversão de Tipo de Dados


In [0]:
df_bronze = df_bronze.withColumn(
    "nota_atendimento", 
    F.when(F.col("nota_atendimento") == "NULL", -1)
    .otherwise(F.col("nota_atendimento").cast(IntegerType()))
)

# Distribuição das notas
display(df_bronze.groupBy("nota_atendimento").count().orderBy("nota_atendimento"))

nota_atendimento,count
-1,2700
2,8530
3,12155
4,5250
5,1365


#### 3. Tratamento de Valores Nulos
Mediana calculada: 3.0

Registros afetados: 2.700 (9%)

Estratégia: Substituição pela mediana

In [0]:
notas_validas = df_bronze.filter(F.col("nota_atendimento") != -1)
mediana = notas_validas.approxQuantile("nota_atendimento", [0.5], 0.01)[0]

print(f"Mediana das notas válidas: {mediana}")

# Substituir -1 pela mediana
df_bronze = df_bronze.withColumn(
    "nota_atendimento",
    F.when(F.col("nota_atendimento") == -1, mediana)
     .otherwise(F.col("nota_atendimento"))
)

# Distribuição das notas
display(df_bronze.groupBy("nota_atendimento").count().orderBy("nota_atendimento"))

Mediana das notas válidas: 3.0


nota_atendimento,count
2.0,8530
3.0,14855
4.0,5250
5.0,1365


#### 5. Ordenação
Ordenado por id_pesquisa para melhor organização

In [0]:
# Ordenar por id_pesquisa
df_bronze = df_bronze.orderBy(F.col("id_pesquisa"))

display(df_bronze.limit(20))

id_pesquisa,id_chamado,nota_atendimento,data_ingestao
1,1,3.0,2025-11-21T14:38:41.734Z
2,2,2.0,2025-11-21T14:38:41.734Z
3,3,3.0,2025-11-21T14:38:41.734Z
4,4,3.0,2025-11-21T14:38:41.734Z
5,28317,3.0,2025-11-21T14:38:41.734Z
6,5,2.0,2025-11-21T14:38:41.734Z
7,6,3.0,2025-11-21T14:38:41.734Z
8,7,2.0,2025-11-21T14:38:41.734Z
9,8,2.0,2025-11-21T14:38:41.734Z
10,9,3.0,2025-11-21T14:38:41.734Z


### Análise Completa de Qualidade - Pesquisa de Satisfação

In [0]:
# Análise completa de qualidade
qualidade_dados = df_bronze.select([
    F.count("*").alias("total_registros"),
    F.countDistinct("id_chamado").alias("id_chamado_unicos"),
    F.countDistinct("id_pesquisa").alias("id_pesquisa_unicos"),
    F.avg("nota_atendimento").alias("nota_media"),
    F.min("nota_atendimento").alias("nota_minima"),
    F.max("nota_atendimento").alias("nota_maxima")
]).collect()[0]

print("RELATÓRIO DE QUALIDADE:")
print(f"Total de registros: {qualidade_dados['total_registros']}")
print(f"IDs chamado únicos: {qualidade_dados['id_chamado_unicos']}")
print(f"IDs pesquisa únicos: {qualidade_dados['id_pesquisa_unicos']}")
print(f"Nota média: {qualidade_dados['nota_media']:.2f}")
print(f"Range notas: [{qualidade_dados['nota_minima']}, {qualidade_dados['nota_maxima']}]")

# Verificar se há gaps nos IDs
if 'id_chamado' in df_bronze.columns:
    gaps = df_bronze.select(
        F.min("id_chamado").alias("min_id"),
        F.max("id_chamado").alias("max_id"),
        (F.max("id_chamado") - F.min("id_chamado") + 1 - F.count("id_chamado")).alias("gaps_total")
    ).collect()[0]
    print(f"Gaps na sequência de IDs: {gaps['gaps_total']}")

RELATÓRIO DE QUALIDADE:
Total de registros: 30000
IDs chamado únicos: 30000
IDs pesquisa únicos: 30000
Nota média: 2.98
Range notas: [2.0, 5.0]
Gaps na sequência de IDs: 0


#### Métricas Gerais da Base

| Métrica | Valor | Observação |
|---------|-------|------------|
| **Total de Registros** | 30.000 | Base completa sem perdas |
| **IDs Pesquisa Únicos** | 30.000 | 100% de unicidade |
| **IDs Chamado Únicos** | 30.000 | 100% de unicidade |
| **Nota Média** | 2.98 | Escala de 2-5 |
| **Nota Mínima** | 2.0 | Limite inferior da escala |
| **Nota Máxima** | 5.0 | Limite superior da escala |
| **Desvio Padrão** | 0.84 | Baixa variabilidade |
| **Gaps na Sequência** | 0 | Dados sequenciais íntegros |


### Categorização das Notas
- **(1 a 2)**: Insatisfeito
- **3**: Neutro
- **(4 a 5)**: Satisfeito

In [0]:
df_bronze = df_bronze.withColumn(
    "categoria_nota",
    F.when(F.col("nota_atendimento").between(1, 2), "Insatisfeito")
     .when(F.col("nota_atendimento") == 3, "Neutro")
     .when(F.col("nota_atendimento").between(4, 5), "Satisfeito")
     .otherwise("Indefinido")
)

print("Distribuição por categoria:")
display(df_bronze.groupBy("categoria_nota").count().orderBy("count", ascending=False))

Distribuição por categoria:


categoria_nota,count
Neutro,14855
Insatisfeito,8530
Satisfeito,6615


In [0]:
display(df_bronze.limit(10))
df_bronze.printSchema()

id_pesquisa,id_chamado,nota_atendimento,data_ingestao,categoria_nota
1,1,3.0,2025-11-21T14:38:41.734Z,Neutro
2,2,2.0,2025-11-21T14:38:41.734Z,Insatisfeito
3,3,3.0,2025-11-21T14:38:41.734Z,Neutro
4,4,3.0,2025-11-21T14:38:41.734Z,Neutro
5,28317,3.0,2025-11-21T14:38:41.734Z,Neutro
6,5,2.0,2025-11-21T14:38:41.734Z,Insatisfeito
7,6,3.0,2025-11-21T14:38:41.734Z,Neutro
8,7,2.0,2025-11-21T14:38:41.734Z,Insatisfeito
9,8,2.0,2025-11-21T14:38:41.734Z,Insatisfeito
10,9,3.0,2025-11-21T14:38:41.734Z,Neutro


root
 |-- id_pesquisa: integer (nullable = true)
 |-- id_chamado: integer (nullable = true)
 |-- nota_atendimento: double (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- categoria_nota: string (nullable = false)




### Conclusão
- **Dados íntegros**: Sem duplicatas ou gaps
- **Qualidade alta**: Todos os problemas tratados
- **Pronto para análise**: Categorias criadas e dados limpos

### Salvando na silver_credit

In [0]:
# Salvando na silver
# df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{gold_db}.{nome_tabela}")
df_bronze.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.pesquisa_satisfacao")
print("Salvo com sucesso!")

Salvo com sucesso!
